In [4]:
from pybm.examples.predator_prey import  generate_synthetic_data

# Generate model and data from /home/urhp/Documents/PyBM/src/pybm/examples/predator_prey.py
model, components, times = generate_synthetic_data()
print(str(model))

# print initial values of the variables
print("Initial values of the variables:")
for var_name, var in model.vars.items():
    print(f"{var_name}: {var.initial}")  

# set initial values of the constants to good guesses
model.consts["growth_rate_prey"].initial_value = 0.5
model.consts["growth_rate_predator"].initial_value = -0.1
model.consts["predation_rate"].initial_value = 0.05
model.consts["conversion_efficiency"].initial_value = 0.05



# print initial values of the constants
print("\nInitial values of the constants:")
for const_name, const in model.consts.items():
    print(f"{const_name}: {const.initial_value}")


Model(Entities: [],
 Vars: ['n_prey', 'n_predator', 'temperature'],
 Consts: ['growth_rate_prey', 'growth_rate_predator', 'predation_rate', 'conversion_efficiency'])
Initial values of the variables:
n_prey: 40.000492061342996
n_predator: 8.016754323781468
temperature: None

Initial values of the constants:
growth_rate_prey: 0.5
growth_rate_predator: -0.1
predation_rate: 0.05
conversion_efficiency: 0.05


In [5]:
import cma
cma.CMAOptions()

{'AdaptSigma': 'True  # or False or any CMAAdaptSigmaBase class e.g. CMAAdaptSigmaTPA, CMAAdaptSigmaCSA',
 'CMA_active': 'True  # negative update, conducted after the original update',
 'CMA_active_injected': '0  #v weight multiplier for negative weights of injected solutions',
 'CMA_cmean': '1  # learning rate for the mean value',
 'CMA_const_trace': 'False  # normalize trace, 1, True, "arithm", "geom", "aeig", "geig" are valid',
 'CMA_diagonal': '0*100*N/popsize**0.5  # nb of iterations with diagonal covariance matrix, True for always',
 'CMA_diagonal_decoding': '0  # learning rate multiplier for additional diagonal update',
 'CMA_eigenmethod': 'np.linalg.eigh  # or cma.utilities.math.eig or pygsl.eigen.eigenvectors',
 'CMA_elitist': 'False  #v or "initial" or True, elitism likely impairs global search performance',
 'CMA_injections_threshold_keep_len': '1  #v keep length if Mahalanobis length is below the given relative threshold',
 'CMA_mirrors': 'popsize < 6  # values <0.5 are int

In [6]:
from cma import fmin2, CMAEvolutionStrategy, fmin2
from pybm.estimate.int_scipy import get_data_matrix, get_initial_const_ctx, simulate
import numpy as np


vars = model.get_endo_variables()
# get the data, that we want to fit the model to
data = get_data_matrix(*vars, t_eval=times)
# initial context:
initial_ctx = get_initial_const_ctx(model)


def residuals(const_ctx):
    # get predictions
    try:
        sol = simulate(*vars, t_eval=times, const_ctx=const_ctx, max_iter=1000, method='Radau', verbose=0)
    except Exception as e:
        print(e)
        return np.array([float("inf")] * (data.size))
    pred = sol.y # of shape (n_vars, n_time_points)
    try:
        return (pred - data ).ravel()
    except Exception as e:
       # print("Error in residuals calculation")
       # print(f"pred shape: {pred.shape}, data shape: {data.shape}")
        return np.array([float("inf")] * (data.size))

evals = []

def fitness(const_ctx):
    evals.append(const_ctx)
    res = residuals(const_ctx)
    return np.sum(res**2)

bounds = (-5,5)  # bounds for the constants
# slow af
xopt, es = fmin2(fitness, initial_ctx, sigma0=0.5, options={'maxiter': 1000, 'verb_disp': 1, 
                                                            'bounds' : bounds,
                                                            'tolfun' : 1e-1})
#es = CMAEvolutionStrategy(initial_ctx, 0.5)
#es.optimize(fitness, iterations=10)

# ask and tell interface
# es = CMAEvolutionStrategy(initial_ctx, 0.5, {'bounds': bounds, 'tolfun': 1e-1, 'verb_disp': 1, 'popsize' : 5})

#for i in range(10):
#    print(f"Iteration {i+1}")
#    solutions = es.ask()
#    fitness_values = [fitness(sol) for sol in solutions]
#    es.tell(solutions, fitness_values)
#    es.disp()

(4_w,8)-aCMA-ES (mu_w=2.6,w_1=52%) in dimension 4 (seed=823530, Mon Aug  3 11:00:12 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      8 2.816424721437437e+07 1.0e+00 4.34e-01  4e-01  4e-01 0:00.1
    2     16 3.673431324615867e+07 1.3e+00 3.58e-01  3e-01  4e-01 0:00.2
    3     24 3.640162555244854e+07 1.4e+00 3.59e-01  3e-01  3e-01 0:00.3


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [1, 2, 4, 6] are not finite but [np.float64(inf), np.float64(inf), np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:12 2026 class=CMAEvolutionStrategy method=ask)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [2, 4, 7] are not finite but [np.float64(inf), np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:12 2026 class=CMAEvolutionStrategy method=ask iteration=1)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [2, 4] are not finite but [np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:12 2026 class=CMAEvolutionStrategy method=ask iteration=2)
  warnings.warn(msg + 

    4     32 3.645099806196489e+07 1.5e+00 4.05e-01  3e-01  4e-01 0:00.4
    5     40 3.647711928887454e+07 1.6e+00 4.06e-01  3e-01  4e-01 0:00.5
    6     48 3.637904509975296e+07 1.9e+00 3.69e-01  2e-01  4e-01 0:00.6
    7     56 3.686673543246891e+07 1.9e+00 3.19e-01  2e-01  3e-01 0:00.6


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [1, 2, 5] are not finite but [np.float64(inf), np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:12 2026 class=CMAEvolutionStrategy method=ask iteration=4)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [2, 3, 6] are not finite but [np.float64(inf), np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:13 2026 class=CMAEvolutionStrategy method=ask iteration=5)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [0, 2] are not finite but [np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:13 2026 class=CMAEvolutionStrategy method=ask iteration=6)
  warnings.warn(msg + ' (time=

    8     64 3.637189336998300e+07 2.1e+00 3.08e-01  2e-01  3e-01 0:00.7
    9     72 3.636836982057875e+07 2.1e+00 2.72e-01  1e-01  3e-01 0:00.8
   10     80 3.637062434255507e+07 2.1e+00 2.63e-01  1e-01  2e-01 0:00.9
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
   11     88 3.639064528202111e+07 2.5e+00 2.20e-01  9e-02  2e-01 0:01.0
   12     96 3.635435171824302e+07 3.0e+00 2.07e-01  7e-02  2e-01 0:01.0
   13    104 3.636742096761334e+07 3.3e+00 2.28e-01  7e-02  3e-01 0:01.1
   14    112 3.641084959355649e+07 4.1e+00 2.29e-01  7e-02  3e-01 0:01.2
   15    120 3.638981966248830e+07 4.9e+00 2.46e-01  7e-02  3e-01 0:01.2
   16    128 3.638146631691717e+07 5.6e+00 2.90e-01  7e-02  4e-01 0:01.3
   17    136 3.633803837390453e+07 7.4e+00 2.96e-01  7e-02  4e-01 0:01.3
   18    144 3.583003789150666e+07 7.8e+00 3.21e-01  7e-02  4e-01 0:01.4
   19    152 3.512962686706094e+07 8.1e+00 2.92e-01  6e-02  4e-01 0:01.5
   20    160 3.616091654944742e+07 8.4e+00 3.21e-01 

...

  243   1944 1.929457457357799e+07 1.1e+03 1.42e-01  1e-05  1e-02 0:17.0
  244   1952 1.929334830376808e+07 1.2e+03 1.79e-01  1e-05  2e-02 0:17.1
  245   1960 1.929444313567643e+07 1.5e+03 1.97e-01  1e-05  2e-02 0:17.2
  246   1968 1.929348498269023e+07 1.6e+03 2.98e-01  2e-05  4e-02 0:17.3
  247   1976 1.929367575409611e+07 1.5e+03 2.83e-01  2e-05  4e-02 0:17.4
  248   1984 1.929433940794550e+07 1.8e+03 2.42e-01  2e-05  3e-02 0:17.5
  249   1992 1.929359576481182e+07 2.0e+03 2.33e-01  2e-05  3e-02 0:17.6
  250   2000 1.929431346491502e+07 2.3e+03 2.09e-01  1e-05  3e-02 0:17.7
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
  251   2008 1.929354946638341e+07 2.1e+03 2.13e-01  1e-05  3e-02 0:17.8
  252   2016 1.929504426600622e+07 2.3e+03 2.09e-01  1e-05  3e-02 0:17.9
  253   2024 1.929246226353560e+07 2.6e+03 2.08e-01  1e-05  3e-02 0:18.0
  254   2032 1.929295991482655e+07 2.8e+03 2.47e-01  2e-05  3e-02 0:18.1
  255   2040 1.929296473115208e+07 2.1e+03 2.74e-01 

/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [1] are not finite but [np.float64(inf)]. (time=Aug  3 11:00:33 2026 class=CMAEvolutionStrategy method=ask iteration=280)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +


  283   2264 1.928380482750269e+07 5.9e+03 1.52e+00  1e-04  1e-01 0:20.8
  284   2272 1.928893281865363e+07 5.7e+03 1.49e+00  1e-04  1e-01 0:20.9


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [1] are not finite but [np.float64(inf)]. (time=Aug  3 11:00:33 2026 class=CMAEvolutionStrategy method=ask iteration=282)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +


  285   2280 1.928024063077823e+07 5.9e+03 1.36e+00  1e-04  1e-01 0:21.1
  286   2288 1.928228510735601e+07 6.2e+03 1.30e+00  9e-05  1e-01 0:21.2
  287   2296 1.928162313087949e+07 5.5e+03 1.08e+00  7e-05  1e-01 0:21.2
  288   2304 1.928248278723408e+07 6.0e+03 1.04e+00  6e-05  9e-02 0:21.3
  289   2312 1.927927442164247e+07 5.8e+03 8.32e-01  5e-05  6e-02 0:21.4
  290   2320 1.927911791369788e+07 6.0e+03 7.48e-01  5e-05  5e-02 0:21.5
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
  291   2328 1.927616375486290e+07 6.7e+03 1.03e+00  6e-05  8e-02 0:21.6
  292   2336 1.927619125147006e+07 6.4e+03 1.15e+00  7e-05  8e-02 0:21.7
  293   2344 1.927473916025518e+07 6.2e+03 1.24e+00  7e-05  9e-02 0:21.8
  294   2352 1.926987078026338e+07 6.7e+03 1.35e+00  9e-05  8e-02 0:21.9
  295   2360 1.927250980414622e+07 7.7e+03 1.53e+00  1e-04  9e-02 0:22.0
  296   2368 1.927275774603409e+07 8.2e+03 1.47e+00  1e-04  9e-02 0:22.1
  297   2376 1.926739071374027e+07 7.5e+03 1.75e+00 

In [7]:
xopt, es

(array([ 0.05324053, -4.84236926, -0.01127791, -3.65179754]),
 <cma.evolution_strategy.CMAEvolutionStrategy at 0x7fc545209d30>)